# Preprocessing of RGI 11 (Italy and Austria) point mass balance data

This notebook builds the Italian and Austrian stake (point mass balance, PMB) dataset used by MassBalanceMachine, from the WGMS Fluctuations of Glaciers database (FoG, version 10.5904/wgms-fog-2025-02b; data from 2006 onwards).

**Pipeline**
1. Load the WGMS point mass balance data and keep stakes in Austria and Italy; add the missing RGI ID of Careser.
2. Rename the columns to the MBM convention and fix one wrong elevation.
3. Fix dates: wrong years (1012 instead of 2012), highly uncertain dates (± 182 days) and two inconsistent periods.
4. Download OGGM glacier directories and sample topographic variables at each stake; drop stakes outside the RGI outline.
5. Merge stakes closer than 10 m (same glacier, year and period).
6. Add the sky-view factor (SVF) from the per-glacier grids made by `1.4 Create_grids` (part 1).
7. Re-assign stake IDs by elevation.
8. Final cleaning: drop outliers on Lupo, drop winter balances ending in July, relabel some winter balances as annual.
9. Save `IT_AT_wgms_dataset_all.csv` to `cfg.dataPath/WGMS/Italy_Austria/csv/`.

## Setting up:

In [ ]:
import os, sys
sys.path.append(os.path.join(os.getcwd(), '../../../../')) # Add root of repo to import MBM

import pandas as pd
import warnings
import massbalancemachine as mbm
import pyproj
import seaborn as sns
import matplotlib.pyplot as plt
import xarray as xr
import geopandas as gpd
from oggm import utils
from tqdm import tqdm
from cmcrameri import cm


# Helper functions shared across regions (OGGM, stake QC, SVF)
from papers.Transformer_TF.scripts.oggm import initialize_oggm_glacier_directories, export_oggm_grids, merge_pmb_with_oggm_data
from papers.Transformer_TF.scripts.pmb_processing import rename_stakes_by_elevation, check_point_ids_contain_glacier, remove_close_points, check_multiple_rgi_ids

from papers.Transformer_TF.scripts.pmb_processing import add_svf_from_rgi_zarr, plot_missing_svf_for_all_glaciers, add_svf_nearest_valid
# Italy/Austria-specific helpers (fix_uncertain_dates, check_period_consistency)
from papers.Transformer_TF.PMB_sources.RGI_11_Italy_and_Austria.scripts.italy_austria_preprocess import *

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

# Call config file (data paths, seed, etc.)
cfg = mbm.ItalyAustriaConfig()

## Load stakes into one DataFrame

The data come directly from the WGMS Fluctuations of Glaciers (FoG) database, version 10.5904/wgms-fog-2025-02b.
- `mass_balance_point.csv`: point mass balance measurements (one row per stake and period).
- `glacier.csv`: glacier metadata, used to map the WGMS `glacier_id` to its RGI v6 ID (`rgi60_ids`).

In [ ]:
# Raw WGMS FoG files (shared by all WGMS-based regions)
path_PMB_WGMS_raw = "WGMS/raw/"

df_stakes = pd.read_csv(cfg.dataPath + path_PMB_WGMS_raw +
                        'mass_balance_point.csv')
df_it_at_RGIId = pd.read_csv(cfg.dataPath + path_PMB_WGMS_raw + 'glacier.csv')

# Filter df_stakes to include only rows where country is AT or IT
df_stakes = df_stakes[df_stakes['country'].isin(['AT',
                                                 'IT'])].reset_index(drop=True)

# Create a mapping dictionary from id to rgi60_ids
id_to_rgi_map = dict(zip(df_it_at_RGIId['id'], df_it_at_RGIId['rgi60_ids']))

# Add the RGIId column to the filtered DataFrame using glacier_id instead of id
df_stakes['RGIId'] = df_stakes['glacier_id'].map(id_to_rgi_map)

# Display glacier names with NaN RGIId
display(f"Number of rows with NaN RGIId: {df_stakes['RGIId'].isna().sum()}")
display(df_stakes[df_stakes['RGIId'].isna()]['glacier_name'].unique())

# Only Careser glacier has NaN RGIIds as only RGIId_50 are listed in the csv file.
# (Careser, Careser Centrale, Careser Occidentale and Careser Orientale)

## find RGIId_60 for Careser glaciers
glacier_outline = gpd.read_file(
    cfg.dataPath + "RGI_v6/RGI_11_CentralEurope/11_rgi60_CentralEurope.shp")

# Search by name
careser_glacier = glacier_outline[
    glacier_outline['Name'].notna()
    & glacier_outline['Name'].str.contains('CARESER', case=False)]
display(careser_glacier[['RGIId', 'Name']])

# RGIId_60 and 50 are the same: RGI50-11.01834 and RGI60-11.01834, add to df
# (applied to every glacier without RGI ID, i.e. all four Careser entries)
for glacier_id in df_stakes[df_stakes['RGIId'].isna()]['glacier_id'].unique():
    df_stakes.loc[df_stakes['glacier_id'] == glacier_id,
                  'RGIId'] = 'RGI60-11.01834'

display(f"Number of rows with NaN RGIId: {df_stakes['RGIId'].isna().sum()}")

In [ ]:
# Free-text remarks of the WGMS data providers (for inspection)
display(df_stakes['remarks'].unique())

#### Rename columns

In [ ]:
# Select and rename columns to the MBM naming convention
df_stakes_renamed = df_stakes.rename(
    columns={
        'point_id': 'POINT_ID',
        'latitude': 'POINT_LAT',
        'longitude': 'POINT_LON',
        'elevation': 'POINT_ELEVATION',
        'begin_date': 'FROM_DATE',
        'end_date': 'TO_DATE',
        'balance': 'POINT_BALANCE',
        'glacier_name': 'GLACIER',
        'year': 'YEAR',
        'country': 'COUNTRY',
        'balance_code': 'PERIOD'
    })

# Create new POINT_ID column: <glacier>_<year>_<WGMS measurement id>_<country>
df_stakes_renamed['POINT_ID'] = (df_stakes_renamed['GLACIER'] + '_' +
                                 df_stakes_renamed['YEAR'].astype(str) + '_' +
                                 df_stakes['id'].astype(str) + '_' +
                                 df_stakes_renamed['COUNTRY'])
# Only keep relevant columns in df
df_stakes_renamed = df_stakes_renamed[[
    'POINT_ID', 'POINT_LAT', 'POINT_LON', 'POINT_ELEVATION', 'FROM_DATE',
    'TO_DATE', 'POINT_BALANCE', 'GLACIER', 'PERIOD', 'RGIId', 'YEAR',
    'begin_date_unc', 'end_date_unc'
]]

# Remove rows with NaN values in POINT_LAT, POINT_LON, and POINT_ELEVATION
df_stakes_renamed = df_stakes_renamed.dropna(
    subset=['POINT_LAT', 'POINT_LON', 'POINT_ELEVATION'])

# change date format to YYYYMMDD
df_stakes_renamed['FROM_DATE'] = df_stakes_renamed['FROM_DATE'].astype(
    str).str.replace('-', '')
df_stakes_renamed['TO_DATE'] = df_stakes_renamed['TO_DATE'].astype(
    str).str.replace('-', '')

# Add data modification column to keep track of mannual changes
df_stakes_renamed['DATA_MODIFICATION'] = ''

display(df_stakes_renamed.head(2))

## General fixes
One stake has a wrong elevation of 296 m instead of 2960 m.

In [ ]:
# Check if any entry anywhere is NaN
display(df_stakes_renamed[df_stakes_renamed.isna().any(axis=1)])

# One stake has a wrong elevation of 296 instead of 2960
display(df_stakes_renamed[df_stakes_renamed['POINT_ID'] ==
                          'VERNAGT F._2013_15124_AT'])
df_stakes_renamed.loc[df_stakes_renamed['POINT_ID'] ==
                      'VERNAGT F._2013_15124_AT', 'POINT_ELEVATION'] = 2960
df_stakes_renamed.loc[
    df_stakes_renamed['POINT_ID'] == 'VERNAGT F._2013_15124_AT',
    'DATA_MODIFICATION'] = 'Elevation corrected from 296 to 2960 m'
display(df_stakes_renamed[df_stakes_renamed['POINT_ID'] ==
                          'VERNAGT F._2013_15124_AT'])

### Date fixes
1. Dates that cannot be parsed (all `10120508` on Malavalle, i.e. year 1012) are corrected to 2012.
2. Dates with an uncertainty of ~182 days (Oe. Wurten K., Vernagt F. and Grand Etret) are always entered as early July. They are set to the hydrological year: 1 October for start dates, 30 September for annual end dates and 30 April for winter end dates (see `fix_uncertain_dates`).
3. `check_period_consistency` flags annual periods that do not last 9–15 months and winter periods that do not last 4–9 months. The two cases found are fixed by hand.

In [ ]:
# Some stakes have the year 1012 instead of 2012 etc. find all these stakes
# (dates that cannot be parsed as YYYYMMDD)
problematic_dates = []
for i, date in enumerate(df_stakes_renamed['FROM_DATE']):
    try:
        pd.to_datetime(str(date), format="%Y%m%d")
    except:
        problematic_dates.append((i, date, 'FROM_DATE'))

for i, date in enumerate(df_stakes_renamed['TO_DATE']):
    try:
        pd.to_datetime(str(date), format="%Y%m%d")
    except:
        problematic_dates.append((i, date, 'TO_DATE'))

print(f"Found {len(problematic_dates)} problematic date entries")
if problematic_dates:
    print(problematic_dates)

# All stakes from same glacier MALAVALLE and date 10120508. Correct the date
df_stakes_renamed.loc[
    df_stakes_renamed['FROM_DATE'] == '10120508',
    'DATA_MODIFICATION'] = 'Date corrected from 10120508 to 20120508'
df_stakes_renamed['FROM_DATE'] = df_stakes_renamed['FROM_DATE'].replace(
    '10120508', '20120508')

In [ ]:
## In the original dataset Glaciers "OE. WURTEN K.", "VERNAGT F." and "GRAND ETRET" have multiple measurements with date_unc 182 or 182.5
## These dates are always entered as start of july, correct them to the hydrological year:
## 01.10 for start dates, 30.09 for annual end dates, 30.04 for winter end dates

display(df_stakes_renamed[(df_stakes_renamed['begin_date_unc'] >= 182)
                          | (df_stakes_renamed['end_date_unc'] >= 182)])

# Update the DATA_MODIFICATION column for these rows
uncertain_date_mask = (df_stakes_renamed['begin_date_unc']
                       >= 182) | (df_stakes_renamed['end_date_unc'] >= 182)
df_stakes_renamed.loc[
    uncertain_date_mask,
    'DATA_MODIFICATION'] = "Dates corrected due to high uncertainty (~= 182 days)"

# Update dates
df_stakes_renamed = fix_uncertain_dates(df_stakes_renamed)

display(df_stakes_renamed[(df_stakes_renamed['begin_date_unc'] >= 182)
                          | (df_stakes_renamed['end_date_unc'] >= 182)])

# Remove _unc columns, were only needed for fixing uncertain dates
df_stakes_renamed = df_stakes_renamed[[
    'POINT_ID', 'POINT_LAT', 'POINT_LON', 'POINT_ELEVATION', 'FROM_DATE',
    'TO_DATE', 'POINT_BALANCE', 'GLACIER', 'PERIOD', 'RGIId', 'YEAR',
    'DATA_MODIFICATION'
]]

In [ ]:
annual_inconsistent, winter_inconsistent = check_period_consistency(
    df_stakes_renamed)

pd.set_option('display.max_rows', None)
display(annual_inconsistent)
display(winter_inconsistent)
pd.reset_option('display.max_rows')

## 2 Cases of inconsistent periods:
# 1. HALLSTAETTER G._2024_63282_AT has MONTH_DIFF of 1, unclear whether this is a date error or an actual measurement (since pmb is also lower than other stakes in that year, just remove it)
df_stakes_renamed = df_stakes_renamed.loc[df_stakes_renamed['POINT_ID'] !=
                                          'HALLSTAETTER G._2024_63282_AT']

# 2. GRAND ETRET in Year 2008 goes from 1999 to 2008, assuming this is a date error and changing year to 2007
mask = (df_stakes_renamed['GLACIER']
        == 'GRAND ETRET') & (df_stakes_renamed['YEAR'] == 2008)
df_stakes_renamed.loc[
    mask, 'DATA_MODIFICATION'] = 'FROM_DATE year corrected from 1999 to 2007'
df_stakes_renamed.loc[mask, 'FROM_DATE'] = df_stakes_renamed.loc[
    mask, 'FROM_DATE'].str.replace('1999', '2007')

annual_inconsistent, winter_inconsistent = check_period_consistency(
    df_stakes_renamed)

In [ ]:
# Check if any entry anywhere is NaN
display(df_stakes_renamed[df_stakes_renamed.isna().any(axis=1)])

### Add OGGM data:
Download the OGGM glacier directories (OGGM v1.6, `elev_bands_w_data`) of the stake glaciers, export their gridded data, and sample the topographic variables at the nearest grid cell of each stake. `merge_pmb_with_oggm_data` also flags whether each stake lies within its RGI outline (`within_glacier_shape`).

In [ ]:
rgi_ids_list = df_stakes_renamed.RGIId.unique().tolist()
# initialize OGGM glacier directories (only the stake glaciers)
gdirs, _ = initialize_oggm_glacier_directories(
    cfg,
    rgi_region="11",
    rgi_version="62",
    base_url=
    "https://cluster.klima.uni-bremen.de/~oggm/gdirs/oggm_v1.6/L1-L2_files/2025.6/elev_bands_w_data/",
    log_level='WARNING',
    task_list=None,
    rgi_ids_list=rgi_ids_list)

# Save the OGGM gridded data of these glaciers to disk
export_oggm_grids(cfg, gdirs, rgi_region="11", subset_rgis=rgi_ids_list)

In [ ]:
# Diagnostic: DEM source and grid resolution of the OGGM glacier directories
from oggm import utils
stats = utils.compile_glacier_statistics(gdirs)
print(stats['dem_source'].value_counts())

# grid resolution per glacier
print(pd.Series([gd.grid.dx for gd in gdirs]).describe())

In [ ]:
unique_rgis = df_stakes_renamed['RGIId'].unique()

# Sample topographic variables at each stake and flag stakes within the glacier outline
# (default variables: aspect, slope, topo, hugonnet_dhdt, consensus_ice_thickness, millan_v)
df_stakes_topo = merge_pmb_with_oggm_data(
    df_pmb=df_stakes_renamed,
    gdirs=gdirs,
    rgi_region="11",  # Central Europe
    rgi_version="62")

# Restrict to within glacier shape
df_stakes_topo = df_stakes_topo[df_stakes_topo['within_glacier_shape'] == True]
df_stakes_topo = df_stakes_topo.drop(columns=['within_glacier_shape'])

# Display rows with NaN topography (should be empty)
display(df_stakes_topo[df_stakes_topo["aspect"].isna()])
display(df_stakes_topo[df_stakes_topo["slope"].isna()])
display(df_stakes_topo[df_stakes_topo["topo"].isna()])

# Drop 3 rows where consensus_ice_thickness is NaN
#df_stakes_topo_dropped = df_stakes_topo.dropna(subset=['consensus_ice_thickness'])

### Merge close stakes:
Within each glacier, year and period, stakes closer than 10 m are merged into one point with the mean `POINT_BALANCE` (see `remove_close_points`). Summer balances (and WGMS `index` balances) are removed first.

In [ ]:
from tqdm import tqdm

# Keep only annual and winter balances (drops summer and WGMS 'index' rows)
df_stakes_topo = df_stakes_topo[df_stakes_topo['PERIOD'].isin(['annual', 'winter'])]

# Merge stakes < 10 m apart, glacier by glacier
df_pmb_topo = pd.DataFrame()
for gl in tqdm(df_stakes_topo.GLACIER.unique(), desc='Merging stakes'):
    print(f'-- {gl.capitalize()}:')
    df_gl = df_stakes_topo[df_stakes_topo.GLACIER == gl]
    df_gl_cleaned = remove_close_points(df_gl)
    df_pmb_topo = pd.concat([df_pmb_topo, df_gl_cleaned])
# Drop temporary projected coordinates added by remove_close_points
df_pmb_topo.drop(['x', 'y'], axis=1, inplace=True)
df_pmb_topo.reset_index(inplace=True, drop=True)

### Add sky-view factor:
The SVF is sampled at each stake from the per-glacier grids (`<RGIId>.zarr` in `xr_masked_grids/`). Stakes on a masked (NaN) pixel are first plotted, then filled with the nearest valid SVF pixel.

In [ ]:
# Directory with the per-glacier SVF grids (<RGIId>.zarr)
path_masked_xr = os.path.join(cfg.dataPath,
                              "RGI_v6/RGI_11_CentralEurope/xr_masked_grids")

# Nearest-pixel SVF sampling (NaN if the pixel is masked)
df_pmb_topo_svf = add_svf_from_rgi_zarr(
    df_pmb_topo,
    path_masked_xr,
    rgi_col="RGIId",
    lon_col="POINT_LON",
    lat_col="POINT_LAT",
    svf_var="svf",
    out_col="svf",
)
# Stakes that fell on a masked (NaN) SVF pixel
df_missing = df_pmb_topo_svf[df_pmb_topo_svf["svf"].isna()].copy()
print("Missing SVF points:", len(df_missing))
print("Glaciers affected:", sorted(df_missing["RGIId"].unique()))

# Plot SVF grids with missing-SVF stakes (red) and valid stakes (white) per glacier
plot_missing_svf_for_all_glaciers(
    df_with_svf=df_pmb_topo_svf,
    path_masked_xr=path_masked_xr,
    plot_valid_points=True,
    save_dir=
    None  # or e.g. os.path.join(cfg.dataPath, "diagnostics/svf_missing")
)

In [ ]:
# Sample SVF again; points on masked pixels are filled with the nearest
# valid pixel within max_radius grid cells. This is the SVF that is kept.
df_pmb_topo_svf_new = add_svf_nearest_valid(
    df_pmb_topo,
    path_masked_xr,
    rgi_col="RGIId",
    lon_col="POINT_LON",
    lat_col="POINT_LAT",
    svf_var="svf",
    out_col="svf",
    max_radius=30,  # ~30 grid cells search; adjust if needed
)

print("Missing SVF points after nearest-valid fill:",
      df_pmb_topo_svf_new["svf"].isna().sum())

# Same plot after filling: there should be no red points left
plot_missing_svf_for_all_glaciers(
    df_with_svf=df_pmb_topo_svf_new,
    path_masked_xr=path_masked_xr,
    plot_valid_points=True,
    save_dir=
    None  # or e.g. os.path.join(cfg.dataPath, "diagnostics/svf_missing")
)

### Give new stake IDs:
Stake IDs are re-assigned per glacier as `<GLACIER>_<rank>`, where the rank is the stake's mean elevation (0 = lowest).

In [ ]:
# Rename POINT_IDs to <GLACIER>_<rank>, ranked by elevation
df_pmb_new_ids = rename_stakes_by_elevation(df_pmb_topo_svf_new)

# Check that every POINT_ID contains its glacier name
check_point_ids_contain_glacier(df_pmb_new_ids)

print('Number of winter and annual samples:', len(df_pmb_new_ids))
print('Number of annual samples:',
      len(df_pmb_new_ids[df_pmb_new_ids.PERIOD == 'annual']))
print('Number of winter samples:',
      len(df_pmb_new_ids[df_pmb_new_ids.PERIOD == 'winter']))

# Histogram of mass balance
df_pmb_new_ids['POINT_BALANCE'].hist(bins=20)
plt.xlabel('Mass balance [m w.e.]')

### Final cleaning:
1. Remove winter balances above 2 m w.e. on Lupo.
2. Remove winter measurements that end in July.
3. Relabel "winter" measurements that end in June with a negative balance as annual.

In [ ]:
# Remove implausibly high winter balances (> 2 m w.e.) on Lupo
df_pmb_new_ids = df_pmb_new_ids[~(
    (df_pmb_new_ids.GLACIER == 'LUPO') &
    (df_pmb_new_ids.PERIOD == 'winter') &
    (df_pmb_new_ids.POINT_BALANCE > 2)
)]

In [ ]:
df_pmb_clean = df_pmb_new_ids.copy()

# Ensure YYYYMMDD format
df_pmb_clean["FROM_DATE"] = df_pmb_clean["FROM_DATE"].astype(str).str.zfill(8)
df_pmb_clean["TO_DATE"] = df_pmb_clean["TO_DATE"].astype(str).str.zfill(8)

# Extract months (MM) of start and end of each measurement
df_pmb_clean["MONTH_START"] = df_pmb_clean["FROM_DATE"].str[4:6]
df_pmb_clean["MONTH_END"] = df_pmb_clean["TO_DATE"].str[4:6]


# Helper: print unique start/end months for winter and annual measurements
def print_months(df, label):
    winter = df[df.PERIOD == "winter"]
    annual = df[df.PERIOD == "annual"]

    print(f"\n{label}")
    print("Winter measurement months:")
    print("  Unique start months:", sorted(winter["MONTH_START"].unique()))
    print("  Unique end months:  ", sorted(winter["MONTH_END"].unique()))

    print("\nAnnual measurement months:")
    print("  Unique start months:", sorted(annual["MONTH_START"].unique()))
    print("  Unique end months:  ", sorted(annual["MONTH_END"].unique()))


# --- Before filtering ---
print_months(df_pmb_clean, "Before filtering")

# -----------------------
# Filtering masks (define + count BEFORE filtering)
# -----------------------

# Winter measurements ending in July span the melt season
mask_winter_end_07 = (
    (df_pmb_clean["PERIOD"].astype(str).str.strip().str.lower() == "winter") &
    (df_pmb_clean["MONTH_END"] == "07"))

# counts (on original df)
n_total_removed = int(mask_winter_end_07.sum())
n_winter_end_07 = n_total_removed

# Apply removal
df_pmb_clean = df_pmb_clean.loc[~mask_winter_end_07].copy()

# --- Correct mislabeled winter MB ---
# A "winter" balance ending in June that is negative already includes melt,
# so it is treated as annual
mask_fix = (
    (df_pmb_clean["PERIOD"].astype(str).str.strip().str.lower() == "winter") &
    (df_pmb_clean["MONTH_END"] == "06") & (df_pmb_clean["POINT_BALANCE"] < 0))
n_relabel = int(mask_fix.sum())
df_pmb_clean.loc[mask_fix, "PERIOD"] = "annual"

print(f"\nRemoved {n_total_removed} rows total.\n"
      f"  - winter-end-07 rows removed: {n_winter_end_07}\n"
      f"Relabeled winter -> annual: {n_relabel}")

print_months(df_pmb_clean, "After filtering + relabeling")

## Save final dataframe:

In [ ]:
# Save to csv:
df_pmb_clean.to_csv(os.path.join(cfg.dataPath, "WGMS/Italy_Austria/csv",
                                 'IT_AT_wgms_dataset_all.csv'),
                    index=False)

# Histogram of mass balance
df_pmb_clean['POINT_BALANCE'].hist(bins=20)
plt.xlabel('Mass balance [m w.e.]')

## Data overview:

In [ ]:
# Interactive map of all cleaned stakes, one layer per glacier
from papers.Transformer_TF.scripts.plotting.folium import plot_stakes_folium

m = plot_stakes_folium(df_pmb_clean, color_map=None, layer_control=True)
m

In [ ]:
# Number of measurements per year (top) and per glacier (bottom)
print('Number of winter and annual samples:', len(df_pmb_clean))
print('Number of annual samples:',
      len(df_pmb_clean[df_pmb_clean.PERIOD == 'annual']))
print('Number of winter samples:',
      len(df_pmb_clean[df_pmb_clean.PERIOD == 'winter']))

# Number of measurements per year:
fig, axs = plt.subplots(2, 1, figsize=(20, 15))
ax = axs.flatten()[0]
df_pmb_clean.groupby(['YEAR', 'PERIOD']).size().unstack().plot(
    kind='bar',
    stacked=True,
    color=[mbm.plots.COLOR_ANNUAL, mbm.plots.COLOR_WINTER],
    ax=ax)
ax.set_title('Number of measurements per year for all glaciers')

ax = axs.flatten()[1]
num_gl = df_pmb_clean.groupby(['GLACIER']).size().sort_values()
num_gl.plot(kind='bar', ax=ax)
ax.set_title('Number of total measurements per glacier since 1951')
plt.tight_layout()

In [ ]:
# Remaining NaN per column (only OGGM variables that are not model inputs)
nan_counts = df_pmb_clean.isna().sum().sort_values(ascending=False)
print(nan_counts[nan_counts > 0])